# *Stoic* — Protein Stoichiometry Prediction

**Fast and accurate protein stoichiometry prediction.**

Enter one protein sequence per unique chain type (colon-separated). *Stoic* predicts how many copies of each chain are present in the assembled complex.

1. Run the **Setup** cell (click the play button — the code is hidden).
2. Fill in the **Predict** form and run it.

---

In [ ]:
#@title **Setup** — install dependencies & load model { display-mode: "form" }
#@markdown *Run this cell once. It takes ~1 min on a T4 GPU.*

!pip install -q stoic pandas matplotlib

import time

import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from stoic.model import Stoic

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = Stoic.from_pretrained("PickyBinders/stoic").to(device).eval()
print("Model loaded.")

In [ ]:
#@title **Predict stoichiometry** { display-mode: "form" }

#@markdown Enter protein sequences separated by **colons** (`:`).  
#@markdown One sequence per unique chain type (max 26).
sequences_input = "MKTLLILTLFLAIAASSASA:MGSSHHHHHHSSGLVPR" #@param {type:"string"}

#@markdown ---
#@markdown **Options**
top_n = 3 #@param {type:"slider", min:1, max:10, step:1}
return_weights = True #@param {type:"boolean"}

# --- parse & validate ---
sequences = [s.strip() for s in sequences_input.split(":") if s.strip()]
assert 1 <= len(sequences) <= 26, "Enter between 1 and 26 unique chains."

chain_labels = [chr(ord("A") + i) for i in range(len(sequences))]

print(f"Predicting stoichiometry for {len(sequences)} chain(s)…")
for label, seq in zip(chain_labels, sequences):
    preview = seq[:50] + "…" if len(seq) > 50 else seq
    print(f"  Chain {label}: {preview}")

# --- predict ---
start = time.time()
with torch.no_grad():
    raw = model.predict_stoichiometry(
        sequences, top_n=top_n, return_residue_weights=return_weights
    )
elapsed = time.time() - start

if return_weights:
    results, residue_predictions = raw
else:
    results = raw

# --- results table ---
header = "| Rank | " + " | ".join(f"Chain {l}" for l in chain_labels) + " | Stoichiometry | Score | Probability |"
separator = "|------|" + "|".join("------" for _ in chain_labels) + "|---------------|-------|-------------|"
rows = []
for rank, candidate in enumerate(results, 1):
    copies = [candidate.get(seq, 0) for seq in sequences]
    stoich = "".join(f"{l}<sub>{c}</sub>" for l, c in zip(chain_labels, copies))
    score = candidate.get("rank", 0)
    prob = candidate.get("probability", 0)
    row = f"| {rank} | " + " | ".join(str(c) for c in copies) + f" | {stoich} | {score:.2f} | {prob:.2e} |"
    rows.append(row)

table_md = "\n".join([header, separator] + rows)

legend = "\n\n**Sequences:**\n"
for label, seq in zip(chain_labels, sequences):
    preview = seq[:50] + "…" if len(seq) > 50 else seq
    legend += f"- **Chain {label}**: `{preview}`\n"

display(Markdown(table_md + legend))
print(f"\n⏱ Runtime: {elapsed:.2f}s")

# --- residue weights plot ---
if return_weights:
    pred_residues = residue_predictions["pred_residues"]
    attention_mask = residue_predictions["attention_mask"]
    seqs = residue_predictions["sequences"]

    records = []
    for i, seq in enumerate(seqs):
        mask = ~(attention_mask[i].astype(bool))
        weights = pred_residues[i][mask]
        for pos, w in enumerate(weights, 1):
            records.append({"Position": pos, "Weight": float(w), "Chain": f"Chain {chain_labels[i]}"})

    df = pd.DataFrame(records)

    fig, ax = plt.subplots(figsize=(12, 4))
    for chain, grp in df.groupby("Chain"):
        ax.plot(grp["Position"], grp["Weight"], label=chain, alpha=0.8)
    ax.set_xlabel("Residue Position")
    ax.set_ylabel("Weight")
    ax.set_title("Residue-level Interface Weights")
    ax.legend()
    plt.tight_layout()
    plt.show()